# Sentinel-2 Cross-Seasonal Dataset Pipeline
This notebook prepares matched patch pairs from Sentinel-2 winter and summer satellite images.

### Key Processing Steps:
1. **Image Loading**: Convert Sentinel-2 `.jp2` files into RGB numpy arrays.
2. **Sliding Window Patching**: Crop image pairs into 512x512 patches with a stride of 256.
3. **Black Border Filtering**: Discard patches near black image borders or water bodies (`mean brightness < 25`).
4. **Cloud Filtering**: Discard patches with excessive cloud cover (> 40% threshold).
5. **Split Structure**: Save corresponding patches under `data/processed/{train|val}/{winter|summer}/`.

In [ ]:
import os
import cv2
import numpy as np

def load_image(path: str) -> np.ndarray:
    """
    Loads an image from file and converts it to RGB format.
    """
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not load image at path: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


In [ ]:
def create_matching_dataset(winter_img: np.ndarray, summer_img: np.ndarray, 
                            save_dir: str = "data/processed", 
                            patch_size: int = 512, 
                            stride: int = 256):
    """
    Slices matched satellite image pairs into smaller patches,
    filtering out dark borders and heavy cloud coverage.
    """
    os.makedirs(f"{save_dir}/winter", exist_ok=True)
    os.makedirs(f"{save_dir}/summer", exist_ok=True)

    h, w, _ = winter_img.shape
    patch_count = 0
    skipped_clouds = 0
    skipped_black = 0

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):

            w_patch = winter_img[y: y + patch_size, x: x + patch_size]
            s_patch = summer_img[y: y + patch_size, x: x + patch_size]

            # Filter black borders / dark regions
            if np.mean(w_patch) < 25 or np.mean(s_patch) < 25:
                skipped_black += 1
                continue

            # Calculate cloud ratios
            white_pixels_s = np.sum(np.all(s_patch > 210, axis=-1))
            white_pixels_w = np.sum(np.all(w_patch > 245, axis=-1))

            cloud_ratio_s = white_pixels_s / (patch_size ** 2)
            cloud_ratio_w = white_pixels_w / (patch_size ** 2)
                             
            # Filter heavy cloud cover
            if cloud_ratio_s > 0.40 or cloud_ratio_w > 0.40:
                skipped_clouds += 1
                continue

            patch_name = f"patch_{patch_count:04d}.png"
            
            # Save corresponding patches
            cv2.imwrite(f"{save_dir}/winter/{patch_name}", cv2.cvtColor(w_patch, cv2.COLOR_RGB2BGR))
            cv2.imwrite(f"{save_dir}/summer/{patch_name}", cv2.cvtColor(s_patch, cv2.COLOR_RGB2BGR))

            patch_count += 1

    print(f"patch count: {patch_count}")
    print(f"skipped because of clouds: {skipped_clouds}")
    print(f"skipped because of black borders: {skipped_black}")


In [ ]:
# Run dataset creation for train and val splits
image_pairs = [
    {
        "winter": "data/raw/winter1.jp2",
        "summer": "data/raw/summer1.jp2",
        "split": "val"
    },
    {
        "winter": "data/raw/winter2.jp2",
        "summer": "data/raw/summer2.jp2",
        "split": "train"
    }
]

for pair in image_pairs:
    print(f"{pair['split']}:")
    winter_img = load_image(pair['winter'])
    summer_img = load_image(pair['summer'])
    create_matching_dataset(
        winter_img=winter_img,
        summer_img=summer_img,
        save_dir=f"data/processed/{pair['split']}",
        patch_size=512,
        stride=256
    )
    print('-'*60)
